<a href="https://colab.research.google.com/github/dylankam/fyp-model1/blob/main/finetune/fyp_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
# 1. Install Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install pyngrok flask

In [2]:
from unsloth import FastLanguageModel
from flask import Flask, request, jsonify
from pyngrok import ngrok
from google.colab import userdata
import json
import torch

# 2. Load the BASE Instruct Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

# 3. Define the ORIGINAL Many-Shot Prompt
system_prompt = (
    "You are a Cartesian mapping model for a humanoid robot \n"
    "Input: A JSON intent containing 'use_hand' ('left', 'right', or 'both'), a description, and duration.\n"
    "Output: A JSON block containing a 'keyframes' array. Each keyframe represents a waypoint in the animation.\n"
    "\n"
    "KEYFRAME & use_hand RULES:\n"
    "- You MUST output a list of 1 or more keyframes inside a 'keyframes' array.\n"
    "- Only include more than 1 keyframe if the description implies a clear sequential action (e.g., 'first', 'then', 'afterwards').\n"
    "- Each keyframe MUST have a 'time_fraction' (a float between 0.1 and 1.0) indicating when in the audio this pose is reached.\n"
    "- TIMING (CRITICAL): Humans gesture quickly then hold. For a static pose, use low time_fraction to reach the target fast, then add a second keyframe at time_fraction=1.0 with the SAME position to hold it. Do NOT use time_fraction=1.0 as the only keyframe for a static gesture.\n"
    "- TIMING & INTERPOLATION (CRITICAL): The robot moves sequentially. Movement toward Keyframe 2 only begins exactly when Keyframe 1's time_fraction is reached.\n"
    "- ANCHORING (HOLDING POSES): If a hand is NOT included in a keyframe, it will completely FREEZE and hold its last known position up to that time_fraction.\n"
    "- 'use_hand' dictates which hands are given new target coordinates at this specific moment.\n"
    "- If use_hand is 'right': Output ONLY right_hand keys. OMIT left_hand keys (the left arm will freeze in place).\n"
    "- If use_hand is 'left': Output ONLY left_hand keys. OMIT right_hand keys (the right arm will freeze in place).\n"
    "- If use_hand is 'both': Output keys for both hands (both arms move to new targets).\n"
    "\n"
    "NORMALIZED SPATIAL CONSTRAINTS (CRITICAL):\n"
    "- Do NOT output meters. You must output normalized float coordinates between -1.0 and 1.0.\n"
    "- X (Forward/Back): 0.0 is the lower torso. 1.0 is maximum reach forward.\n"
    "- Y (Left/Right): 0.0 is the center of the lower torso. +/- 1.0 is maximum reach outward. Left hand uses positive Y, Right hand uses negative Y.\n"
    "- Z (Up/Down): -1.0 is resting at the waist. 0.0 is the center of the lower torso. 1.0 is the height of the face/head.\n"
    "- Orientations: MUST be one of the following (flat of palm faces the stated direction):\n"
    "    'palms_forward'  = palm faces away from robot chest (toward audience).\n"
    "    'palms_backward' = palm faces toward robot's own chest.\n"
    "    'palms_up'       = palm faces ceiling.\n"
    "    'palms_down'     = palm faces floor.\n"
    "    'palms_in'       = palm faces body midline (palms face each other).\n"
    "    'palms_out'      = palm faces away from body midline.\n"
    "- Fingers: MUST be one of ['forward', 'backward', 'up', 'down', 'left', 'right'].\n"
    "\n"
    "PHYSICAL ARM LIMITS:\n"
    "- The IK solver is independent per arm and cannot avoid inter-arm collisions. Keep arms well separated.\n"
    "\n"
    "ORTHOGONALITY RULE (ANATOMY LIMITS):\n"
    "Palms and fingers CANNOT point along the same axis. They must be 90 degrees apart.\n"
    "- If palm is 'forward'/'backward', fingers MUST be 'up', 'down', 'left', or 'right'.\n"
    "- If palm is 'up'/'down', fingers MUST be 'forward', 'backward', 'left', or 'right'.\n"
    "- If palm is 'in'/'out', fingers MUST be 'forward', 'backward', 'up', or 'down'.\n"
    "\n"
    "SPATIAL GUIDELINES:\n"
    "- 'Stop': X=0.15, Z=0.5, orientation: 'palms_forward', fingers: 'up'.\n"
    "- 'Welcome' or 'Present': X=0.05, active Y=+/- 0.1, Z=-1.0, orientation: 'palms_up', fingers: 'forward'.\n"
    "\n"
    "Example Output 1 (Single / Static Target):\n"
    "```json\n"
    '{"keyframes": [\n'
    '  {"time_fraction": 0.3, "use_hand": "right", "right_hand_pos": [0.15, -0.07, 0.05], "right_orientation": "palms_forward", "right_fingers": "up"},\n'
    '  {"time_fraction": 1.0, "use_hand": "right", "right_hand_pos": [0.15, -0.07, 0.05], "right_orientation": "palms_forward", "right_fingers": "up"}\n'
    '], "duration": 1.6}\n'
    "```\n"
    "Example Output 2 (Sequential Action):\n"
    "```json\n"
    '{"keyframes": [\n'
    '  {"time_fraction": 0.5, "use_hand": "right", "right_hand_pos": [0.15, -0.07, 0.05], "right_orientation": "palms_forward", "right_fingers": "up"},\n'
    '  {"time_fraction": 1.0, "use_hand": "both", "right_hand_pos": [0.15, -0.07, 0.05], "right_orientation": "palms_forward", "right_fingers": "up", "left_hand_pos": [0.15, 0.07, 0.05], "left_orientation": "palms_forward", "left_fingers": "up"}\n'
    '], "duration": 4.2}\n'
    "```"
)

# 4. Authenticate Ngrok
try:
    my_ngrok_token = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(my_ngrok_token)
except Exception as e:
    print("Error: Could not find NGROK_TOKEN in Colab Secrets! Make sure it is added.")

# 5. Build the Flask API Server
app = Flask(__name__)

@app.route('/generate_gesture', methods=['POST'])
def generate_gesture():
    # Receive the JSON intent from the local Brain Server
    incoming_data = request.json
    user_content = json.dumps(incoming_data)

    # Format the prompt
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate the baseline model's response
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True, temperature=0.1)
    decoded = tokenizer.batch_decode(outputs)[0]

    # Clean the payload
    final_json_str = decoded.split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip()

    # Send it back to your local laptop script
    return jsonify({"gesture_payload": final_json_str})

# 6. Start the Server
public_url = ngrok.connect(5000).public_url
print(f"\n YOUR BASELINE API URL IS: {public_url}/generate_gesture \n")

app.run(port=5000)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8B-Instruct-bnb-4bit as a legacy tokenizer.



 YOUR BASELINE API URL IS: https://cortex-thermal-lurk.ngrok-free.dev/generate_gesture 

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
